## 1. Setup

In [120]:
from pathlib import Path
from datetime import date
from hashlib import sha256
from importlib import reload
import sys

import openpyxl
import pulp as pl
import ipywidgets as widgets
from IPython.display import FileLink, display

BASE_DIR = Path.cwd()
if not (BASE_DIR / "scheduling_dataset.xlsx").exists():
    BASE_DIR = BASE_DIR / "Scheduling"
BASE_DIR = BASE_DIR.resolve()
sys.path.insert(0, str(BASE_DIR))
import dashboard

reload(dashboard)

WORKBOOK = BASE_DIR / "scheduling_dataset.xlsx"
OUTPUT_DIR = BASE_DIR / "output"
workbook = openpyxl.load_workbook(WORKBOOK, data_only=True)


def rows(sheet):
    values = list(workbook[sheet].values)
    return [
        dict(zip(values[0], row))
        for row in values[1:]
        if any(v is not None for v in row)
    ]


doctors = [
    {
        "id": r["DoctorID"],
        "name": r["Name"],
        "seniority": r["Seniority"],
        "max_shifts": r["MaxShifts"],
    }
    for r in rows("Doctors")
]
shifts = [
    {"id": r["ShiftID"], "name": r["Name"], "start": r["Start"], "end": r["End"]}
    for r in rows("Shifts")
]
days = [
    {
        "day": r["Day"],
        "date": str(r["Date"])[:10],
        "weekday": r["Weekday"],
        "is_weekend": bool(r["IsWeekend"]),
        "week": r["Day"] // 7 + 1,
    }
    for r in rows("Horizon")
]
demand = [
    {"day": r["Day"], "shift_id": r["ShiftID"], "required": r["Required"]}
    for r in rows("Demand")
]
unavailable = [
    {"doctor_id": r["DoctorID"], "day": r["Day"]} for r in rows("Unavailability")
]
source = {
    "file": WORKBOOK.name,
    "sha256": sha256(WORKBOOK.read_bytes()).hexdigest(),
    "sheets": ["Doctors", "Shifts", "Horizon", "Demand", "Unavailability"],
    "classification": "Teaching dataset",
    "readme": [
        " | ".join(str(v) for v in row if v is not None)
        for row in workbook["README"].values
        if any(v is not None for v in row)
    ],
}
workbook.close()
data = {
    "doctors": doctors,
    "shifts": shifts,
    "days": days,
    "demand": demand,
    "unavailability": unavailable,
    "source": source,
}
data

def show_switch(name, description, default=True):
    previous = globals().get(name)
    value = previous.value if previous is not None else default
    if previous is not None:
        previous.close()
    control = widgets.Checkbox(value=value, description=description,
                               style={"description_width": "initial"})
    display(control)
    return control


## 2. Constraints

**Assignment variables.** There are 6 doctors ($p\in P=\{D1,\ldots,D6\}$), 14 days ($d\in D=\{0,\ldots,13\}$), and 3 shifts ($s\in S=\{\mathrm{Day},\mathrm{Evening},\mathrm{Night}\}$).

$$x_{pds}\in\{0,1\}.$$

$x_{pds}=1$ means doctor $p$ works shift $s$ on day $d$; otherwise it is 0.

Use the switches below, then re-run **Solve**. The model is rebuilt from the current settings.

In [121]:
P = [p["id"] for p in doctors]
D = [d["day"] for d in days]
S = [s["id"] for s in shifts]
constraint_builders = {}

def create_model():
    model = pl.LpProblem("Physician_Scheduling", pl.LpMinimize)
    x = {(p, d, s): pl.LpVariable(f"x_{p}_{d}_{s}", cat=pl.LpBinary)
         for p in P for d in D for s in S}
    return model, x

In [122]:
coverage_enabled = show_switch('coverage_enabled', 'Enforce exact coverage')

Checkbox(value=True, description='Enforce exact coverage', style=CheckboxStyle(description_width='initial'))

**Exact coverage.** Assign exactly 1 doctor to each of the 42 day–shift combinations.

$$\sum_{p\in P}x_{pds}=1\qquad\forall d\in D,\ s\in S.$$

In [123]:
def add_coverage(model, x):
    for row in demand:
        model += pl.lpSum(x[p, row["day"], row["shift_id"]] for p in P) == row["required"]

constraint_builders["coverage"] = add_coverage

In [124]:
capacity_enabled = show_switch('capacity_enabled', 'Enforce total capacity')

Checkbox(value=True, description='Enforce total capacity', style=CheckboxStyle(description_width='initial'))

**Total capacity.** The maximum numbers of shifts for D1–D6 are $C=(8,8,8,8,7,7)$.

$$\sum_{d\in D}\sum_{s\in S}x_{pds}\le C_p\qquad\forall p\in P.$$

In [125]:
def add_capacity(model, x):
    for p in doctors:
        model += pl.lpSum(x[p["id"], d, s] for d in D for s in S) <= p["max_shifts"]

constraint_builders["capacity"] = add_capacity

In [126]:
daily_enabled = show_switch('daily_enabled', 'Enforce one shift per day')

Checkbox(value=True, description='Enforce one shift per day', style=CheckboxStyle(description_width='initial')…

**Daily limit.** Each doctor can start at most 1 shift per day.

$$\sum_{s\in S}x_{pds}\le1\qquad\forall p\in P,\ d\in D.$$

The shifts are 08:30–16:00, 16:00–23:00 and 23:00–08:30 the next day. With these times, the daily limit prevents overlapping assignments; no extra minimum rest period is imposed.

In [127]:
def add_daily_limit(model, x):
    for p in P:
        for d in D:
            model += pl.lpSum(x[p, d, s] for s in S) <= 1

constraint_builders["daily_limit"] = add_daily_limit

In [128]:
availability_enabled = show_switch('availability_enabled', 'Respect unavailable dates')

Checkbox(value=True, description='Respect unavailable dates', style=CheckboxStyle(description_width='initial')…

**Unavailability.** These doctor–day pairs cannot be assigned any shift:

$$U=\{(D1,10),(D3,5),(D3,6),(D5,0),(D5,1),(D6,12)\}.$$

$$x_{pds}=0\qquad\forall(p,d)\in U,\ s\in S.$$

In [129]:
def add_unavailability(model, x):
    for row in unavailable:
        for s in S:
            model += x[row["doctor_id"], row["day"], s] == 0

constraint_builders["unavailability"] = add_unavailability

In [130]:
senior_night_enabled = show_switch('senior_night_enabled', 'Prohibit senior night shifts')

Checkbox(value=True, description='Prohibit senior night shifts', style=CheckboxStyle(description_width='initia…

**Night eligibility.** Senior doctors D1 and D2 cannot work nights.

$$x_{p,d,\mathrm{Night}}=0\qquad\forall p\in\{D1,D2\},\ d\in D.$$

In [131]:
def add_senior_night_restriction(model, x):
    for p in doctors:
        if p["seniority"] == "Senior":
            for d in D:
                model += x[p["id"], d, "Night"] == 0

constraint_builders["senior_night"] = add_senior_night_restriction

In [132]:
senior_weekends_allowed = show_switch('senior_weekends_allowed', 'Allow seniors to work weekends')

Checkbox(value=False, description='Allow seniors to work weekends', style=CheckboxStyle(description_width='ini…

**Senior weekend eligibility.** When the switch is on, D1 and D2 may work weekends. When it is off, prohibit all their shifts on weekend days $D_W=\{5,6,12,13\}$:

$$x_{pds}=0\qquad\forall p\in\{D1,D2\},\ d\in D_W,\ s\in S.$$

In [133]:
def prohibit_senior_weekends(model, x):
    for p in doctors:
        if p["seniority"] == "Senior":
            for d in days:
                if d["is_weekend"]:
                    for s in S:
                        model += x[p["id"], d["day"], s] == 0

# This restriction is applied when the permission switch is OFF.
constraint_builders["senior_weekends_allowed"] = prohibit_senior_weekends

In [134]:
fairness_enabled = show_switch('fairness_enabled', 'Balance workloads')

Checkbox(value=True, description='Balance workloads', style=CheckboxStyle(description_width='initial'))

**Fairness bounds.** Define total shifts $T_p$, night shifts $N_p$ and weekend shifts $W_p$ on days $D_W=\{5,6,12,13\}$.

$$T_p=\sum_{d,s}x_{pds},\qquad N_p=\sum_d x_{p,d,\mathrm{Night}},\qquad W_p=\sum_{d\in D_W,s}x_{pds}.$$

For each measure $k\in\{T,N,W\}$, bound the relevant doctors' workloads and define their spread:

$$L_k\le k_p\le H_k,\qquad\Delta_k=H_k-L_k.$$

Total-shift bounds apply to all 6 doctors. Night bounds compare doctors allowed to work nights; weekend bounds compare doctors allowed to work weekends. Turning fairness off requests a feasible schedule without balancing workloads.

In [135]:
def build_objectives(model, x, settings):
    night_doctors = [p for p in doctors if not settings["senior_night"] or p["seniority"] == "Junior"]
    weekend_doctors = [p for p in doctors if settings["senior_weekends_allowed"] or p["seniority"] == "Junior"]
    if not settings["fairness"]:
        return [("Feasibility only", pl.LpAffineExpression())], night_doctors, weekend_doctors

    weekend_days = [d["day"] for d in days if d["is_weekend"]]
    total = {p: pl.lpSum(x[p, d, s] for d in D for s in S) for p in P}
    night = {p["id"]: pl.lpSum(x[p["id"], d, "Night"] for d in D) for p in night_doctors}
    weekend = {p["id"]: pl.lpSum(x[p["id"], d, s] for d in weekend_days for s in S) for p in weekend_doctors}

    def spread(name, counts):
        if not counts:
            return pl.LpAffineExpression()
        low = pl.LpVariable(f"{name}_minimum", 0, len(D)*len(S), cat=pl.LpInteger)
        high = pl.LpVariable(f"{name}_maximum", 0, len(D)*len(S), cat=pl.LpInteger)
        for count in counts.values():
            model.addConstraint(count >= low)
            model.addConstraint(count <= high)
        return high - low

    night_label = "Junior night shift spread" if settings["senior_night"] else "Night shift spread"
    objectives = [("Total shift spread", spread("total", total)),
                  (night_label, spread("night", night)),
                  ("Weekend shift spread", spread("weekend", weekend))]
    return objectives, night_doctors, weekend_doctors

## 3. Solve

Change any switches above and run this cell again. When workload balancing is on, minimize the three spreads in order, preserving each earlier optimum:

$$\operatorname{lexmin}(\Delta_T,\Delta_N,\Delta_W).$$

Solve with CBC, then generate `output/dashboard.html` using `dashboard.py`.

In [136]:
rule_settings = {
    "coverage": coverage_enabled.value,
    "capacity": capacity_enabled.value,
    "daily_limit": daily_enabled.value,
    "unavailability": availability_enabled.value,
    "senior_night": senior_night_enabled.value,
    "senior_weekends_allowed": senior_weekends_allowed.value,
    "fairness": fairness_enabled.value,
}

# Always rebuild: turning a switch off must also remove the old constraint.
model, x = create_model()
for name, add_constraint in constraint_builders.items():
    enabled = not rule_settings[name] if name == "senior_weekends_allowed" else rule_settings[name]
    if enabled:
        add_constraint(model, x)
objectives, night_doctors, weekend_doctors = build_objectives(model, x, rule_settings)

solver = pl.PULP_CBC_CMD(
    msg=False, timeLimit=60, threads=1, gapRel=0, options=["randomSeed 42"]
)
stages = []
for i, (name, objective) in enumerate(objectives):
    model.setObjective(objective)
    model.solve(solver)
    if model.status != pl.LpStatusOptimal or model.sol_status != pl.LpSolutionOptimal:
        raise RuntimeError(f"No proven optimal solution: {pl.LpStatus[model.status]}")
    optimum = round(pl.value(objective) or 0)
    stages.append({"name": name, "status": "Optimal", "optimum": optimum})
    model += objective == optimum, f"preserve_stage_{i+1}"
    print(f"{name}: {optimum}")

result, dashboard_path = dashboard.save_schedule(
    data, x, stages, night_doctors, OUTPUT_DIR,
    rule_settings=rule_settings, weekend_doctors=weekend_doctors
)
print(f"\n{len(result['assignments'])} shifts assigned; all active constraint checks passed.")
for p in doctors:
    assigned = [a for a in result["assignments"] if a["doctor_id"] == p["id"]]
    print(
        f"{p['name']:12} Total: {len(assigned)}  Nights: {sum(a['shift_id']=='Night' for a in assigned)}"
        f"  Weekend: {sum(a['is_weekend'] for a in assigned)}"
    )
display(FileLink(str(dashboard_path.relative_to(Path.cwd()))))

Total shift spread: 0
Junior night shift spread: 1
Weekend shift spread: 2

42 shifts assigned; all active constraint checks passed.
Dr. Adams    Total: 7  Nights: 0  Weekend: 0
Dr. Brown    Total: 7  Nights: 0  Weekend: 0
Dr. Carter   Total: 7  Nights: 3  Weekend: 2
Dr. Diaz     Total: 7  Nights: 4  Weekend: 3
Dr. Evans    Total: 7  Nights: 4  Weekend: 4
Dr. Foster   Total: 7  Nights: 3  Weekend: 3


/Users/jiaokan/Workshop/AIRM/Scheduling/output/dashboard.html